In [2]:
from typing import List
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.output_parsers.datetime import DatetimeOutputParser
from langchain_classic.output_parsers.boolean import BooleanOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.exceptions import OutputParserException
from langchain_classic.output_parsers import OutputFixingParser
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [4]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
)

## Output Parsers

**String Parser**

In [5]:
llm.invoke("hello")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c73dd82b09', 'id': 'chatcmpl-ED2VMYnFMEO3Xw5XOW4vB4F89161C', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0042f-3ab0-7572-a908-d62c111eb1b8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [6]:
llm.invoke("hello").content

'Hello! How can I assist you today?'

In [7]:
parser = StrOutputParser()

In [8]:
parser.invoke(
    llm.invoke("hello")
)

'Hello! How can I assist you today?'

### Other Parsers

**Datetime**

In [9]:
llm.invoke(
    "Output a random datetime in %Y-%m-%dT%H:%M:%S.%fZ. "
    "Don't say anything else"
)

AIMessage(content='2023-10-05T14:23:45.123456Z', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 33, 'total_tokens': 49, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b1b99e3b92', 'id': 'chatcmpl-ED2VeNjvP1DYNI38laEyzEwwcFyg6', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0042f-82a3-74a2-b128-3eec0197392a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 33, 'output_tokens': 16, 'total_tokens': 49, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [10]:
parser = DatetimeOutputParser()

In [11]:
parser.invoke(
    llm.invoke(
        "Output a random datetime in %Y-%m-%dT%H:%M:%S.%fZ. "
        "Don't say anything else"
    )
)

datetime.datetime(2023, 10, 5, 14, 23, 45, 123456)

**Boolean**

In [12]:
llm.invoke(
    "Are you an AI? YES or NO only"
)

AIMessage(content='YES', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1, 'prompt_tokens': 16, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f344a168a1', 'id': 'chatcmpl-ED2VsMW2QYkZbK0AOl6gTRbLT5Hzs', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0042f-bb7b-7272-b726-b8e168a7cd9e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 1, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [13]:
parser = BooleanOutputParser()

In [14]:
parser.invoke(
    input=llm.invoke(
        "Are you an AI? YES or NO only"
    )
)

True

In [15]:
parser.invoke(
    input=llm.invoke(
        "Are you Human? YES or NO only"
    )
)

False

## Structured

**Dict Schema**

In [16]:
from typing_extensions import Annotated, TypedDict

class UserInfo(TypedDict):
    """User's info."""
    name: Annotated[str, "", "User's name. Defaults to ''"]
    country: Annotated[str, "", "Where the user lives. Defaults to ''"]


In [17]:
llm_with_structure = llm.with_structured_output(UserInfo)

In [18]:
llm_with_structure.invoke(
    "My name is Henrique, and I am from Brazil"
)

{'name': 'Henrique', 'country': 'Brazil'}

In [19]:
llm_with_structure.invoke(
    "The sky is blue"
)

{'name': '', 'country': ''}

In [20]:
llm_with_structure.invoke(
    "Hello, my name is the same as the capital of the U.S.  "
    "But I'm from a country where we usually associate with kangaroos"
)

{'name': 'Washington', 'country': 'Australia'}

**Pydantic**

In [21]:
from pydantic import BaseModel, Field

class PydanticUserInfo(BaseModel):
    """User's info."""
    name: Annotated[str, Field(description="User's name. Defaults to ''", default=None)]
    country: Annotated[str, Field(description="Where the user lives. Defaults to ''", default=None, )]

In [22]:
llm_with_structure = llm.with_structured_output(PydanticUserInfo)

In [23]:
structured_output = llm_with_structure.invoke("The sky is blue")

In [24]:
structured_output

PydanticUserInfo(name='', country='')

In [25]:
print(structured_output.name)

In [26]:
print(structured_output.country)

In [27]:
structured_output = llm_with_structure.invoke(
    "Hello, my name is the same as the capital of the U.S.  "
    "But I'm from a country where we usually associate with kangaroos"
)

In [28]:
structured_output

PydanticUserInfo(name='Washington', country='Australia')

## Dealing with Errors

In [29]:
class Performer(BaseModel):
    """Filmography info about an actor/actress"""
    name: Annotated[str, Field(description="name of an actor/actress")]
    film_names: Annotated[List[str], Field(description="list of names of films they starred in")]

In [30]:
llm_with_structure = llm.with_structured_output(Performer)

In [31]:
response = llm_with_structure.invoke(
    "Generate the filmography for Scarlett Johansson. Top 5 only"
)
response

Performer(name='Scarlett Johansson', film_names=['Lost in Translation (2003)', 'The Avengers (2012)', 'Her (2013)', 'Lucy (2014)', 'Marriage Story (2019)'])

**Fixing Parser**

In [34]:
response.model_dump_json()

'{"name":"Scarlett Johansson","film_names":["Lost in Translation (2003)","The Avengers (2012)","Her (2013)","Lucy (2014)","Marriage Story (2019)"]}'

In [32]:
parser = PydanticOutputParser(pydantic_object=Performer)

In [35]:
parser.parse(response.model_dump_json())

Performer(name='Scarlett Johansson', film_names=['Lost in Translation (2003)', 'The Avengers (2012)', 'Her (2013)', 'Lucy (2014)', 'Marriage Story (2019)'])

In [37]:
misformatted_result = "{'name': 'Scarlett Johansson', 'film_names': ['The Avengers']}"

In [38]:
try:
    parser.parse(misformatted_result)
except OutputParserException as e:
    print(e)

Invalid json output: {'name': 'Scarlett Johansson', 'film_names': ['The Avengers']}
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 


In [39]:
new_parser = OutputFixingParser.from_llm(parser=parser, llm=llm)

In [40]:
new_parser.parse(misformatted_result)

Performer(name='Scarlett Johansson', film_names=['The Avengers', 'Lost in Translation', 'Marriage Story', 'Black Widow'])